# Experiment 05 — Probabilistic Forecasting

**Goal:** Produce prediction intervals using a **quantile BiLSTM**
that outputs P10, P50, and P90 simultaneously.

The model reuses the **same pipeline** (split, preprocessing, windowing)
as the deterministic experiments, making this a methodological extension
rather than a separate experiment.

**Evaluation:**
- Pinball loss (P10, P50, P90)
- PICP (Prediction Interval Coverage Probability) — nominal 90 %
- MPIW (Mean Prediction Interval Width)
- Interval Score

In [ ]:
# ── Cell 1: Environment Setup ────────────────────────────────────
from pathlib import Path
import subprocess, sys

PROJECT_ROOT = Path("/kaggle/working/stlf-entso-2026")

if not PROJECT_ROOT.exists():
    subprocess.run(
        ["git", "clone",
         "https://github.com/AlvinHarist/stlf-entso-2026.git",
         str(PROJECT_ROOT)],
        check=True,
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [ ]:
# ── Cell 2: Imports ──────────────────────────────────────────────
import platform, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml
import tensorflow as tf

from src.utils.seed import set_seed
from src.data.load_data import load_dataset
from src.data.preprocessing import chronological_split, fit_preprocessor, transform_data, inverse_y
from src.data.windowing import create_train_windows, create_evaluation_windows
from src.models.probabilistic import build_quantile_bilstm
from src.training.trainer import train_model
from src.evaluation.point_metrics import compute_all_metrics
from src.evaluation.probabilistic_metrics import compute_all_probabilistic_metrics

print(f"Python: {platform.python_version()} | TF: {tf.__version__}")

In [ ]:
# ── Cell 3: Configuration ────────────────────────────────────────
CONFIG_PATH = PROJECT_ROOT / "configs" / "baseline.yaml"
with open(CONFIG_PATH) as f:
    config = yaml.safe_load(f)

SEED       = config["seed"]
LOOKBACK   = config["windowing"]["lookback"]
HORIZON    = config["windowing"]["horizon"]
TARGET_COL = config["data"]["target_col"]
UNITS      = config["model"]["units"]
DROPOUT    = config["model"]["dropout"]
LR         = config["model"]["learning_rate"]
EPOCHS     = config["training"]["epochs"]
BATCH_SIZE = config["training"]["batch_size"]
PATIENCE   = config["training"]["patience"]
QUANTILES  = config["probabilistic"]["quantiles"]

# Data path
KAGGLE_DATA_DIR = Path("/kaggle/input/stlf-entso-2026")
DATA_PATH = None
if KAGGLE_DATA_DIR.exists():
    for p in KAGGLE_DATA_DIR.rglob("*.csv"):
        if "combined_AT" in p.name:
            DATA_PATH = p
            break
if DATA_PATH is None:
    for fb in [PROJECT_ROOT / config["data"]["path"], PROJECT_ROOT / "df_combined_AT.csv"]:
        if fb.exists():
            DATA_PATH = fb
            break
if DATA_PATH is None:
    raise FileNotFoundError("Cannot locate df_combined_AT.csv")

RESULTS_DIR = PROJECT_ROOT / "results" / "probabilistic"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print(f"Dataset  : {DATA_PATH}")
print(f"Quantiles: {QUANTILES}")

In [ ]:
# ── Cell 4: Seed & Load Data ─────────────────────────────────────
set_seed(SEED)

df = load_dataset(DATA_PATH, timestamp_col=config["data"]["timestamp_col"])
train_df, val_df, test_df = chronological_split(
    df, config["split"]["train_ratio"], config["split"]["val_ratio"]
)

In [ ]:
# ── Cell 5: Preprocessing ────────────────────────────────────────
feature_cols = [TARGET_COL]

preprocessor = fit_preprocessor(
    train_df, TARGET_COL, feature_cols, use_yeojohnson=False
)

X_tr, y_tr = transform_data(train_df, preprocessor)
X_va, y_va = transform_data(val_df, preprocessor)
X_te, y_te = transform_data(test_df, preprocessor)

In [ ]:
# ── Cell 6: Windowing ────────────────────────────────────────────
Xw_tr, yw_tr = create_train_windows(X_tr, y_tr, LOOKBACK, HORIZON)
Xw_va, yw_va = create_evaluation_windows(X_va, y_va, X_tr, y_tr, LOOKBACK, HORIZON)
Xw_te, yw_te = create_evaluation_windows(X_te, y_te, X_va, y_va, LOOKBACK, HORIZON)

print(f"Windows — Train: {Xw_tr.shape}, Val: {Xw_va.shape}, Test: {Xw_te.shape}")

In [ ]:
# ── Cell 7: Build Quantile Model ────────────────────────────────
model = build_quantile_bilstm(
    lookback=LOOKBACK,
    n_features=Xw_tr.shape[2],
    horizon=HORIZON,
    quantiles=QUANTILES,
    units=UNITS,
    dropout=DROPOUT,
    learning_rate=LR,
)
model.summary()

In [ ]:
# ── Cell 8: Train ────────────────────────────────────────────────
train_result = train_model(
    model,
    Xw_tr, yw_tr,
    Xw_va, yw_va,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    patience=PATIENCE,
    save_dir=RESULTS_DIR,
)

In [ ]:
# ── Cell 9: Predict & Inverse Transform ─────────────────────────
# Output shape: (n_windows, horizon, 3)  →  [P10, P50, P90]
pred_scaled = model.predict(Xw_te)

# Inverse transform each quantile separately
pred_p10_mw = inverse_y(pred_scaled[:, :, 0], preprocessor)
pred_p50_mw = inverse_y(pred_scaled[:, :, 1], preprocessor)
pred_p90_mw = inverse_y(pred_scaled[:, :, 2], preprocessor)
actual_mw   = inverse_y(yw_te, preprocessor)

print(f"Predictions shape: P10={pred_p10_mw.shape}, P50={pred_p50_mw.shape}, P90={pred_p90_mw.shape}")

In [ ]:
# ── Cell 10: Point Metrics (P50 as point forecast) ───────────────
point_metrics = compute_all_metrics(actual_mw, pred_p50_mw)
print("P50 Point Forecast Metrics:")
for k, v in point_metrics.items():
    print(f"  {k}: {v:.4f}")

In [ ]:
# ── Cell 11: Probabilistic Metrics ───────────────────────────────
prob_metrics = compute_all_probabilistic_metrics(
    actual_mw, pred_p10_mw, pred_p50_mw, pred_p90_mw
)

print("\nProbabilistic Metrics:")
for k, v in prob_metrics.items():
    print(f"  {k:25s}: {v:.4f}")

print(f"\n  Nominal coverage (P10-P90): 90%")
print(f"  Empirical coverage (PICP) : {prob_metrics['PICP_90']*100:.1f}%")
diff = abs(prob_metrics['PICP_90'] - 0.9) * 100
if diff < 5:
    print(f"  ✓ Calibration looks reasonable (off by {diff:.1f}pp)")
else:
    print(f"  ✗ Calibration concern (off by {diff:.1f}pp)")

In [ ]:
# ── Cell 12: Prediction Interval Plot ────────────────────────────
# Show a representative week from the test set
n_plot = min(168, len(actual_mw))  # ~7 days

fig, ax = plt.subplots(figsize=(14, 5))
x = range(n_plot)

ax.fill_between(x, pred_p10_mw[:n_plot, 0], pred_p90_mw[:n_plot, 0],
                alpha=0.3, color="steelblue", label="P10–P90 interval")
ax.plot(x, actual_mw[:n_plot, 0], color="black", linewidth=0.8, label="Actual")
ax.plot(x, pred_p50_mw[:n_plot, 0], color="steelblue", linewidth=0.8, label="P50 (median)")

ax.set_title("Probabilistic Forecast — Test Set (h=1, first 7 days)")
ax.set_xlabel("Window index")
ax.set_ylabel("Load (MW)")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
fig.savefig(RESULTS_DIR / "quantile_forecast_interval.png", dpi=150)
plt.show()

In [ ]:
# ── Cell 13: Training Loss Curve ─────────────────────────────────
fig2, ax2 = plt.subplots(figsize=(8, 4))
ax2.plot(train_result["history"]["loss"], label="Train")
ax2.plot(train_result["history"]["val_loss"], label="Validation")
ax2.axvline(train_result["best_epoch"] - 1, color="red", linestyle="--", alpha=0.5,
            label=f"Best epoch ({train_result['best_epoch']})")
ax2.set_title("Quantile BiLSTM — Training Loss (Pinball)")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Pinball Loss")
ax2.legend()
ax2.grid(alpha=0.3)
plt.tight_layout()
fig2.savefig(RESULTS_DIR / "quantile_training_loss.png", dpi=150)
plt.show()

In [ ]:
# ── Cell 14: Save Results ────────────────────────────────────────
result_record = {
    "experiment": "05_probabilistic",
    "model": "QuantileBiLSTM",
    "features": "univariate",
    "lookback": LOOKBACK,
    "horizon": HORIZON,
    "quantiles": QUANTILES,
    "seed": SEED,
    "best_epoch": train_result["best_epoch"],
    "best_val_loss": train_result["best_val_loss"],
    "point_metrics_p50": point_metrics,
    "probabilistic_metrics": prob_metrics,
    "python_version": platform.python_version(),
    "tensorflow_version": tf.__version__,
}

out_path = RESULTS_DIR / f"bilstm_quantile_{HORIZON}h_seed{SEED}.json"
with open(out_path, "w") as f:
    json.dump(result_record, f, indent=2)

# Save predictions as CSV
pred_df = pd.DataFrame({
    "actual_h1": actual_mw[:, 0],
    "p10_h1": pred_p10_mw[:, 0],
    "p50_h1": pred_p50_mw[:, 0],
    "p90_h1": pred_p90_mw[:, 0],
})
pred_df.to_csv(RESULTS_DIR / "predictions.csv", index=False)

print(f"Results saved to {out_path}")
print(f"Predictions saved to {RESULTS_DIR / 'predictions.csv'}")